In [3]:
import pandas as pd
import numpy as np
from src.database.connection import engine


In [4]:

invoice_items = pd.read_sql("SELECT * FROM invoice_items;", engine)
anomaly_labels = pd.read_sql("SELECT * FROM anomaly_labels;", engine)

print(f"Loaded {len(invoice_items):,} invoice items, {len(anomaly_labels):,} anomaly labels.")

Loaded 155,476 invoice items, 200 anomaly labels.


## 10. Feature Engineering


We are building a feature table where each row is a invoice, engineered from item-level signals (quanity/unit_price outlieres, tax_rate deviation), addition with basic invoice levels signals. This table will be used to feed both of our models (supervised and unsupervised) with and without is_anomaly label

Note:
Our Data is heavily Right Skewed and using standard z-score ((X - mean) / std) would distorted by heavily tail we found. Therefore, will use a robust or modified z-score based on median and MAD (median absolute deviation), which doesn't dragged around by the outliers it's supposed to flag unlike to the mean.

In [5]:
# Robust z-score: (x - median) / (1.4826 * MAD)

def robust_zscore(series: pd.Series) -> pd.Series:
    median = series.median()
    mad = (series - median).abs().median()

    if mad == 0:
        mad = 1e-9
    return (series - median) / (1.4826 * mad) # 1.4826 makes MAD comparable to std for normally-distributed data —


invoice_items["quantity_zscore"] = robust_zscore(invoice_items["quantity"])
invoice_items["unit_price_zscore"] = robust_zscore(invoice_items["unit_price"])



invoice_items[["quantity", "quantity_zscore", "unit_price", "unit_price_zscore"]].describe()

,quantity,quantity_zscore,unit_price,unit_price_zscore
count,155476.000000,155476.000000,155476.000000,155476.000000
mean,16.939418,0.163515,34.365929,0.028124
std,12.097826,1.019984,20.256510,0.807019
min,1.000000,-1.180359,0.590000,-1.317508
25%,8.000000,-0.590179,16.980000,-0.664531
50%,15.000000,0.000000,33.660000,0.000000
75%,23.220500,0.693081,50.850000,0.684849
max,447.759000,36.486493,263.300000,9.148852


In [6]:
VALID_TAX_RATES = {0.0, 5.0, 20.0}
invoice_items["invalid_tax_rate"] = ~invoice_items["tax_rate"].isin(VALID_TAX_RATES)

invoice_items["invalid_tax_rate"].value_counts()

invalid_tax_rate
False    155414
True         62
Name: count, dtype: int64

### Finding: Robust z-scores correctly isolate known outliers

Median z-score sits at exactly 0 for both `quantity` and `unit_price`, as expected. The known extreme values stand out clearly: max quantity z-score = 36.49 (raw value 447.76), max unit_price z-score = 9.15 (raw value 263.30). Normal items stay within a tight range (~-1.2 to +0.7).

`invalid_tax_rate` flags 62 items — matches the EDA finding exactly (rate outside {0, 5, 20}).

### ML Implication
Median/MAD-based scaling avoids the outlier-dilutes-itself problem a standard mean/std z-score would have here, given the heavy right tail
found in EDA. These three features are ready to aggregate to invoice level.

## item-level scores up to invoice level
Right now these are per-item features. But our model needs to predict per-invoice anomalies (that's what is_anomaly is labeled on). So we need to collapse item-level scores into invoice-level features.
Since, once bad item can hide in an average of many normal ones we will find out the max z-score among the invoice items instead of average z-score.

In [7]:
invoice_level_features = (
    invoice_items
    .groupby("invoice_id")
    .agg(
        item_count = ("id", "count"),
        max_quantity_zscore=("quantity_zscore", "max"),
        max_unit_price_zscore=("unit_price_zscore", "max"),
        any_invalid_tax_rate=("invalid_tax_rate", "any"),
        num_invalid_tax_rate=("invalid_tax_rate", "sum"),
    )
    .reset_index()
)

invoice_level_features.head()

,invoice_id,item_count,max_quantity_zscore,max_unit_price_zscore,any_invalid_tax_rate,num_invalid_tax_rate
0,0001583c-b617-40df-9ec1-282a6012bfed,27,2.413497,1.714712,False,0
1,0004c0ed-4769-498e-b672-6de8df65adc9,27,2.823250,1.329858,False,0
2,000c0d97-1bda-47f2-9c52-25cd9be7d3d7,6,2.349926,0.931857,False,0
3,000c83b9-3af2-4c92-88e8-75fb015b068d,10,2.942550,1.144602,False,0
4,0013d2e2-2cc2-4a4d-84cd-9fb1665d448c,28,2.708839,1.745389,False,0


In [8]:
# Aggregate item-level features up to invoice level.
invoice_level_features = (
    invoice_items
    .groupby("invoice_id")
    .agg(
        item_count=("id", "count"),
        max_quantity_zscore=("quantity_zscore", "max"),
        max_unit_price_zscore=("unit_price_zscore", "max"),
        any_invalid_tax_rate=("invalid_tax_rate", "any"),
        num_invalid_tax_rate=("invalid_tax_rate", "sum"),
    )
    .reset_index()
)

# Attach the label so we can check how well these features separate
# anomalies, before any model is trained.
invoice_level_features = invoice_level_features.merge(
    anomaly_labels[["invoice_id", "anomaly_type", "is_anomaly"]],
    on="invoice_id",
    how="left"
)

invoice_level_features["is_anomaly"] = (
    invoice_level_features["is_anomaly"].fillna(False).astype(bool)
)
invoice_level_features["anomaly_type"] = (
    invoice_level_features["anomaly_type"].fillna("normal")
)

# Quick check: do anomalous invoices already have higher max z-scores
# and more invalid tax rates, even before the item_count fix below?
invoice_level_features.groupby("is_anomaly")[
    ["max_quantity_zscore", "max_unit_price_zscore", "num_invalid_tax_rate"]
].describe()

max_quantity_zscore                                          \
                         count      mean       std       min       25%   
is_anomaly                                                               
False                   9800.0  1.998583  0.884003 -1.180359  1.416262   
True                     200.0  4.422430  5.573783 -0.718248  1.854765   

                                          max_unit_price_zscore            \
                 50%       75%        max                 count      mean   
is_anomaly                                                                  
False       2.319532  2.701778   2.950897                9800.0  1.240789   
True        2.584859  3.665014  36.486493                 200.0  2.053294   

            ...                     num_invalid_tax_rate                       \
            ...       75%       max                count  mean       std  min   
is_anomaly  ...                                                                 
False       ...  1.552564  2.130642               9800.0  0.00  0.000000  0.0   
True        ...  1.908235  9.148852                200.0  0.31  0.463654  0.0   

                                
            25%  50%  75%  max  
is_anomaly                      
False       0.0  0.0  0.0  0.0  
True        0.0  0.0  1.0  1.0  

[2 rows x 24 columns]

### Finding: item_count confound in max(z-score)

Anomalous invoices score higher on average (max_quantity_zscore 4.42 vs 2.00), but normal invoices already average z ≈ 2.0 — not 0 as expected.

This is an order-statistics effect: taking the max across many items naturally produces a higher value just by chance, even with zero real anomalies. Invoices with more items look more "suspicious" by this feature purely because they had more items to take a max over — not because they're actually anomalous.

### ML Implication
Raw max(z-score) is biased by item_count and needs correcting before it's a trustworthy feature.

In [9]:
invoice_level_features = invoice_level_features.merge(
    invoice_items.groupby("invoice_id").agg(
        num_quantity_outliers=("quantity_zscore", lambda s: (s > 3).sum()),
        num_price_outliers=("unit_price_zscore", lambda s: (s > 3).sum()),
    ).reset_index(),
    on="invoice_id",
    how="left"
)

invoice_level_features.groupby("is_anomaly")[
    ["num_quantity_outliers", "num_price_outliers"]
].describe()

num_quantity_outliers                                            \
                           count   mean       std  min  25%  50%  75%  max   
is_anomaly                                                                   
False                     9800.0  0.000  0.000000  0.0  0.0  0.0  0.0  0.0   
True                       200.0  0.255  0.436955  0.0  0.0  0.0  1.0  1.0   

           num_price_outliers                                            
                        count   mean       std  min  25%  50%  75%  max  
is_anomaly                                                               
False                  9800.0  0.000  0.000000  0.0  0.0  0.0  0.0  0.0  
True                    200.0  0.185  0.389272  0.0  0.0  0.0  0.0  1.0

### Finding: threshold-based counts remove the item_count bias

Replacing max() with a count of items exceeding z > 3 fixes it completely: normal invoices score exactly 0, 100% of the time (mean, std, and max all 0.0). Anomalous invoices: 25.5% have ≥1 quantity outlier, 18.5% have ≥1 price outlier.

### ML Implication
`num_quantity_outliers` / `num_price_outliers` are precise (near-zero false positive rate) but only partially complete individually — expected, since each targets a different injected anomaly type. Combined with `num_invalid_tax_rate`, they should cover more anomalies together than any single feature alone.

In [10]:
invoice_level_features["any_quantity_outlier"] = invoice_level_features["num_quantity_outliers"] > 0
invoice_level_features["any_price_outlier"] = invoice_level_features["num_price_outliers"] > 0

overlap_check = invoice_level_features[invoice_level_features["is_anomaly"]][
    ["any_quantity_outlier", "any_price_outlier", "any_invalid_tax_rate", "anomaly_type"]
]



pd.crosstab(
    overlap_check["anomaly_type"],
    [overlap_check["any_quantity_outlier"], overlap_check["any_price_outlier"], overlap_check["any_invalid_tax_rate"]]
)


any_quantity_outlier False             True 
any_price_outlier    False       True  False
any_invalid_tax_rate False True  False False
anomaly_type                                
price_spike             41     0    37     0
quantity_spike           9     0     0    51
tax_error                0    62     0     0

In [11]:
# Instead of a hardcoded set, treat any tax rate that's rare in the
# dataset as unusual. This adapts automatically to whatever rates the
# business actually uses, rather than assuming fixed values.
rate_frequency = invoice_items["tax_rate"].value_counts(normalize=True)

# A rate used in less than 1% of all items is treated as "unusual" —
# this threshold is a judgment call, worth tuning later.
common_rates = rate_frequency[rate_frequency >= 0.01].index

invoice_items["invalid_tax_rate"] = ~invoice_items["tax_rate"].isin(common_rates)

print("Rates treated as common/valid:")
print(rate_frequency[rate_frequency >= 0.01])

print("\nFlag counts:")
print(invoice_items["invalid_tax_rate"].value_counts())

invoice_items["tax_rate"].value_counts(normalize=True)

Rates treated as common/valid:
tax_rate
20.0    0.800059
0.0     0.099913
5.0     0.099630
Name: proportion, dtype: float64

Flag counts:
invalid_tax_rate
False    155414
True         62
Name: count, dtype: int64


tax_rate
20.0    0.800059
0.0     0.099913
5.0     0.099630
17.0    0.000161
13.0    0.000129
27.0    0.000109
Name: proportion, dtype: float64

In [14]:
import os
print(os.getcwd())

/app


In [15]:
invoice_level_features.to_csv(
    "/app/src/data/invoice_level_features.csv",
    index=False
)

print(f"Saved {len(invoice_level_features)} rows.")

Saved 10000 rows.


## Feature Engineering Summary

Built from EDA findings, using item-level data aggregated to invoice level:

- `max_quantity_zscore`, `max_unit_price_zscore` — robust (median/MAD) z-scores,
  resistant to the outliers they're meant to detect
- `num_quantity_outliers`, `num_price_outliers` — count of items per invoice
  exceeding z > 3; fixes an item_count bias found in the raw max (invoices
  with more items falsely scored higher by chance)
- `invalid_tax_rate` — frequency-based: any rate used in <1% of all items,
  rather than a hardcoded list; generalizes without assuming a fixed set
  of "valid" rates

**Coverage check (crosstab against real anomaly_type):** each feature
catches its intended anomaly type with zero cross-contamination —
tax_error: 100% recall, quantity_spike: 85%, price_spike: 47%. price_spike
is the hardest type to catch with current features.

**Important caveat:** these are "unusual for this business" signals, not
proof of error. A rare-but-legitimate custom rate or price would also get
flagged. This is expected — the goal is surfacing invoices worth a human
review, not automatic rejection.